<a href="https://colab.research.google.com/github/hiteshraos/BIS-LAB/blob/main/1BM24CS117_CYBERSECURITY_GREY_WOLF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier

# 1. Objective / Fitness Function
def fitness_function(feature_mask, X, y, alpha_weight=0.9, beta_weight=0.1):
    # If no features are selected, penalize heavily
    selected_indices = np.where(feature_mask == 1)[0]
    if len(selected_indices) == 0:
        return 1.0 # Worst fitness

    X_sub = X[:, selected_indices]
    clf = RandomForestClassifier(n_estimators=10, random_state=42)
    # Evaluate with 3-fold cross-validation
    accuracy = np.mean(cross_val_score(clf, X_sub, y, cv=3, scoring='accuracy'))

    # Minimize error rate + penalty for number of selected features
    error_rate = 1.0 - accuracy
    feature_ratio = len(selected_indices) / X.shape[1]

    score = alpha_weight * error_rate + beta_weight * feature_ratio
    return score

# 2. Grey Wolf Optimizer (Binary GWO for Feature Selection)
def gwo_feature_selection(X, y, search_agents_no=5, max_iter=10):
    dim = X.shape[1]

    # Initialize alpha, beta, and delta positions
    alpha_pos = np.zeros(dim)
    alpha_score = float("inf")

    beta_pos = np.zeros(dim)
    beta_score = float("inf")

    delta_pos = np.zeros(dim)
    delta_score = float("inf")

    # Initialize population randomly (binary: 0 or 1)
    positions = np.random.randint(0, 2, size=(search_agents_no, dim))

    for it in range(max_iter):
        a = 2.0 - it * (2.0 / max_iter)  # a decreases linearly from 2 to 0

        for i in range(search_agents_no):
            # Ensure at least one feature is selected
            if np.sum(positions[i, :]) == 0:
                positions[i, np.random.randint(0, dim)] = 1

            fitness = fitness_function(positions[i, :], X, y)

            if fitness < alpha_score:
                delta_score = beta_score
                delta_pos = beta_pos.copy()
                beta_score = alpha_score
                beta_pos = alpha_pos.copy()
                alpha_score = fitness
                alpha_pos = positions[i, :].copy()
            elif fitness < beta_score:
                delta_score = beta_score
                delta_pos = beta_pos.copy()
                beta_score = fitness
                beta_pos = positions[i, :].copy()
            elif fitness < delta_score:
                delta_score = fitness
                delta_pos = positions[i, :].copy()

        # Update position of wolves (continuous to binary via sigmoid)
        for i in range(search_agents_no):
            for j in range(dim):
                r1, r2 = np.random.rand(), np.random.rand()
                A1 = 2 * a * r1 - a
                C1 = 2 * r2
                D_alpha = abs(C1 * alpha_pos[j] - positions[i, j])
                X1 = alpha_pos[j] - A1 * D_alpha

                r1, r2 = np.random.rand(), np.random.rand()
                A2 = 2 * a * r1 - a
                C2 = 2 * r2
                D_beta = abs(C2 * beta_pos[j] - positions[i, j])
                X2 = beta_pos[j] - A2 * D_beta

                r1, r2 = np.random.rand(), np.random.rand()
                A3 = 2 * a * r1 - a
                C3 = 2 * r2
                D_delta = abs(C3 * delta_pos[j] - positions[i, j])
                X3 = delta_pos[j] - A3 * D_delta

                # Average position
                X_pos = (X1 + X2 + X3) / 3.0
                # Transfer function (sigmoid) to convert to binary {0, 1}
                sigmoid_val = 1 / (1 + np.exp(-X_pos))
                positions[i, j] = 1 if np.random.rand() < sigmoid_val else 0

        print(f"Iteration {it+1}/{max_iter} - Best Alpha Fitness: {alpha_score:.4f} - Features selected: {np.sum(alpha_pos)}")

    return alpha_pos, np.where(alpha_pos == 1)[0]

# --- Sample Execution ---
if __name__ == "__main__":
    np.random.seed(42)
    # Simulate a network dataset: 100 samples, 8 features
    X_sample = np.random.rand(100, 8)
    y_sample = np.random.randint(0, 2, size=100)

    best_mask, best_features = gwo_feature_selection(X_sample, y_sample, search_agents_no=6, max_iter=5)
    print("\nOptimal Feature Mask (Binary):", best_mask)
    print("Selected Feature Indices:", best_features)


Iteration 1/5 - Best Alpha Fitness: 0.3385 - Features selected: 4
Iteration 2/5 - Best Alpha Fitness: 0.3385 - Features selected: 4
Iteration 3/5 - Best Alpha Fitness: 0.3385 - Features selected: 4
Iteration 4/5 - Best Alpha Fitness: 0.3385 - Features selected: 4
Iteration 5/5 - Best Alpha Fitness: 0.3385 - Features selected: 4

Optimal Feature Mask (Binary): [1 1 0 0 0 0 1 1]
Selected Feature Indices: [0 1 6 7]
